# Complete EDA for MOOCCubeX

This notebook performs memory-safe exploratory data analysis for the personalized educational short-video recommendation project. It reads the full MOOCCubeX core dataset from `/content/drive/MyDrive/DataCon/MOOCCubeX` and stores every reusable output under `/content/drive/MyDrive/DataCon/EDA_outputs`.

Main outputs include file validation, schema samples, entity statistics, course-resource statistics, video duration and caption statistics, complete user-video engagement summaries, concept-video graph statistics, prerequisite summaries, ID coverage checks, data-quality reports, charts, and a final EDA report.

The large JSON files are streamed. Do not replace the streaming functions with `json.load()` or `pandas.read_json()`.


## 1. Mount Drive and check the runtime

Select **Runtime → Change runtime type → T4 GPU** before running. EDA mainly uses the CPU; the T4 will be used during model training.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')


In [ ]:
!pip -q install ijson pyarrow seaborn tqdm psutil


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import csv, gc, json, math, os, shutil, statistics, subprocess, sys, time

import ijson
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import pyarrow as pa
import pyarrow.parquet as pq
import psutil
import torch

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 160)

DRIVE_ROOT = Path('/content/drive/MyDrive/DataCon')
RAW_ROOT = DRIVE_ROOT / 'MOOCCubeX'
OUT_ROOT = DRIVE_ROOT / 'EDA_outputs'
TABLE_ROOT = OUT_ROOT / 'tables'
FIG_ROOT = OUT_ROOT / 'figures'
REPORT_ROOT = OUT_ROOT / 'reports'
LOCAL_ROOT = Path('/content/mooccubex_core')

for folder in [OUT_ROOT, TABLE_ROOT, FIG_ROOT, REPORT_ROOT, LOCAL_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)

FORCE_REBUILD = False
USE_LOCAL_CACHE = True
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
print('System RAM:', round(psutil.virtual_memory().total / 1024**3, 2), 'GB')
print('Output directory:', OUT_ROOT)


## 2. Validate files and create an inventory

In [ ]:
EXPECTED_FILES = [
    'entities/video.json', 'entities/user.json', 'entities/course.json',
    'entities/concept.json', 'relations/user-video.json',
    'relations/concept-video.txt', 'relations/video_id-ccid.txt',
    'prerequisites/psy.json', 'prerequisites/cs.json', 'prerequisites/math.json',
]

inventory_rows = []
for rel in EXPECTED_FILES:
    p = RAW_ROOT / rel
    inventory_rows.append({
        'file': rel, 'exists': p.exists(),
        'size_mb': round(p.stat().st_size / 1024**2, 2) if p.exists() else 0,
        'partial_download': Path(str(p) + '.aria2').exists(),
    })

inventory = pd.DataFrame(inventory_rows)
display(inventory)
inventory.to_csv(TABLE_ROOT / 'core_file_inventory.csv', index=False)

missing = inventory.loc[~inventory.exists, 'file'].tolist()
partial = inventory.loc[inventory.partial_download, 'file'].tolist()
if missing or partial:
    raise FileNotFoundError(f'Missing={missing}; incomplete={partial}')
print('All required files exist and have no aria2 partial-download markers.')


## 3. Optional local cache

Copying the core files from mounted Drive to Colab local disk makes repeated parsing much faster. Persistent outputs are still written to Drive. If local disk space is insufficient, set `USE_LOCAL_CACHE = False` and rerun this cell.


In [ ]:
def source_path(relative_path):
    return (LOCAL_ROOT if USE_LOCAL_CACHE else RAW_ROOT) / relative_path

if USE_LOCAL_CACHE:
    selected = EXPECTED_FILES
    required = sum((RAW_ROOT / x).stat().st_size for x in selected)
    free = shutil.disk_usage('/content').free
    print(f'Needed: {required/1024**3:.2f} GB; local free: {free/1024**3:.2f} GB')
    if required > free * 0.90:
        print('Insufficient local storage; reading directly from Drive.')
        USE_LOCAL_CACHE = False
    else:
        for rel in tqdm(selected, desc='Caching core files'):
            src, dst = RAW_ROOT / rel, LOCAL_ROOT / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            if not dst.exists() or dst.stat().st_size != src.stat().st_size:
                shutil.copy2(src, dst)
        print('Local cache ready.')
print('Reading from:', LOCAL_ROOT if USE_LOCAL_CACHE else RAW_ROOT)


## 4. Streaming and utility functions

In [ ]:
def first_nonspace_byte(path):
    with Path(path).open('rb') as f:
        while True:
            b = f.read(1)
            if not b: return b''
            if not b.isspace(): return b

def stream_json(path):
    # Supports a JSON array, JSON Lines, or a top-level JSON object.
    path = Path(path)
    first = first_nonspace_byte(path)
    if first == b'[':
        with path.open('rb') as f:
            yield from ijson.items(f, 'item')
    elif first == b'{':
        # MOOCCubeX entity and behavior files are normally JSON Lines.
        try:
            with path.open('r', encoding='utf-8') as f:
                for line in f:
                    line = line.strip()
                    if line: yield json.loads(line)
        except json.JSONDecodeError:
            with path.open('rb') as f:
                for key, value in ijson.kvitems(f, ''):
                    yield {'_key': key, '_value': value}
    else:
        raise ValueError(f'Unknown JSON format: {path}')

def sample_records(path, n=3):
    out = []
    for obj in stream_json(path):
        out.append(obj)
        if len(out) >= n: break
    return out

def scalar_missing(value):
    return value is None or value == '' or value == [] or value == {}

def savefig(name):
    path = FIG_ROOT / name
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches='tight')
    plt.show()
    print('Saved:', path)

def safe_quantiles(values):
    s = pd.Series(values, dtype='float64')
    return s.quantile([0, .25, .5, .75, .9, .95, .99, 1]).rename_axis('quantile').reset_index(name='value')

def parse_pair_line(line):
    parts = line.rstrip('\n').split('\t')
    if len(parts) < 2: parts = line.strip().split()
    return (parts[0], parts[1]) if len(parts) >= 2 else (None, None)

def preview(name, rel, n=2):
    records = sample_records(source_path(rel), n)
    print(f'\n{name}:')
    for r in records: print(json.dumps(r, ensure_ascii=False)[:1500])
    return records


## 5. Preview actual schemas

In [ ]:
schema_samples = {}
for name, rel in {
    'video':'entities/video.json', 'user':'entities/user.json',
    'course':'entities/course.json', 'concept':'entities/concept.json',
    'user_video':'relations/user-video.json', 'prerequisite_cs':'prerequisites/cs.json'
}.items():
    schema_samples[name] = preview(name, rel)

with (REPORT_ROOT / 'schema_samples.json').open('w', encoding='utf-8') as f:
    json.dump(schema_samples, f, ensure_ascii=False, indent=2)


## 6. Entity-level EDA

This scans course, video, concept, and user entities. Detailed reusable video and course tables are saved as Parquet files.


In [ ]:
entity_specs = {
    'video': ('entities/video.json', ['ccid','name','start','end','text']),
    'course': ('entities/course.json', ['id','name','about','field','prerequisites','resource']),
    'concept': ('entities/concept.json', ['id','name']),
    'user': ('entities/user.json', ['id','gender','school','year_of_birth','course_order','enroll_time']),
}

entity_summary_rows = []
entity_missing_rows = []
video_rows, course_rows = [], []
gender_counts, birth_year_counts, user_course_count_hist = Counter(), Counter(), Counter()
concept_name_missing = 0

for entity, (rel, expected_fields) in entity_specs.items():
    count = 0
    missing = Counter()
    observed_keys = Counter()
    for obj in tqdm(stream_json(source_path(rel)), desc=f'Scanning {entity}'):
        count += 1
        for key in obj: observed_keys[key] += 1
        for field in expected_fields:
            if scalar_missing(obj.get(field)): missing[field] += 1

        if entity == 'video':
            starts, ends, texts = obj.get('start') or [], obj.get('end') or [], obj.get('text') or []
            numeric_ends = [float(x) for x in ends if isinstance(x, (int,float)) or str(x).replace('.','',1).isdigit()]
            video_rows.append({
                'ccid': obj.get('ccid'), 'name': obj.get('name'),
                'duration_seconds': max(numeric_ends) if numeric_ends else np.nan,
                'subtitle_sentences': len(texts),
                'subtitle_characters': sum(len(str(x)) for x in texts),
                'timing_length_match': len(starts) == len(ends) == len(texts),
            })
        elif entity == 'course':
            resources = obj.get('resource') or []
            resource_types = Counter(r.get('resource_type') for r in resources if isinstance(r, dict))
            course_rows.append({
                'course_id': obj.get('id'), 'course_name': obj.get('name'),
                'num_resources': len(resources), 'num_videos': resource_types.get('video',0),
                'num_exercises': resource_types.get('exercise',0),
                'num_fields': len(obj.get('field') or []),
                'has_prerequisite_text': bool(obj.get('prerequisites')),
            })
        elif entity == 'user':
            gender_counts[str(obj.get('gender') or 'missing')] += 1
            yob = str(obj.get('year_of_birth') or '')[:4]
            if yob.isdigit(): birth_year_counts[int(yob)] += 1
            user_course_count_hist[len(obj.get('course_order') or [])] += 1
        elif entity == 'concept':
            if not (obj.get('name') or obj.get('id')): concept_name_missing += 1

    entity_summary_rows.append({'entity':entity, 'records':count, 'observed_fields':len(observed_keys)})
    for field in expected_fields:
        entity_missing_rows.append({'entity':entity, 'field':field, 'missing':missing[field], 'missing_pct':100*missing[field]/max(count,1)})
    pd.DataFrame(observed_keys.most_common(), columns=['field','records_present']).to_csv(TABLE_ROOT/f'{entity}_field_presence.csv', index=False)

entity_summary = pd.DataFrame(entity_summary_rows)
entity_missing = pd.DataFrame(entity_missing_rows)
video_df = pd.DataFrame(video_rows)
course_df = pd.DataFrame(course_rows)

entity_summary.to_csv(TABLE_ROOT/'entity_counts.csv', index=False)
entity_missing.to_csv(TABLE_ROOT/'entity_missingness.csv', index=False)
video_df.to_parquet(TABLE_ROOT/'video_eda.parquet', index=False)
course_df.to_parquet(TABLE_ROOT/'course_eda.parquet', index=False)
pd.DataFrame(gender_counts.items(), columns=['gender','users']).to_csv(TABLE_ROOT/'user_gender_counts.csv', index=False)
pd.DataFrame(birth_year_counts.items(), columns=['birth_year','users']).sort_values('birth_year').to_csv(TABLE_ROOT/'user_birth_year_counts.csv', index=False)
pd.DataFrame(user_course_count_hist.items(), columns=['courses','users']).sort_values('courses').to_csv(TABLE_ROOT/'user_course_count_hist.csv', index=False)

display(entity_summary)
display(entity_missing)
display(video_df.describe(include='all'))
display(course_df.describe(include='all'))


## 7. Video and course charts

In [ ]:
valid_duration = video_df.loc[video_df.duration_seconds.between(1, video_df.duration_seconds.quantile(.99)), 'duration_seconds']
plt.figure(figsize=(9,5)); sns.histplot(valid_duration/60, bins=60)
plt.xlabel('Video duration (minutes)'); plt.title('Video duration distribution (up to 99th percentile)'); savefig('video_duration_distribution.png')

short_thresholds = [60, 180, 300, 600, 900]
short_stats = pd.DataFrame({'threshold_seconds': short_thresholds})
short_stats['videos'] = short_stats.threshold_seconds.map(lambda x: int(video_df.duration_seconds.between(1,x).sum()))
short_stats['percentage'] = 100*short_stats.videos/len(video_df)
short_stats.to_csv(TABLE_ROOT/'short_video_candidates.csv', index=False)
display(short_stats)

plt.figure(figsize=(9,5)); sns.histplot(course_df.num_videos.clip(upper=course_df.num_videos.quantile(.99)), bins=50)
plt.xlabel('Videos per course'); plt.title('Course video counts'); savefig('videos_per_course.png')


## 8. Relation and graph EDA

This builds mappings between platform video IDs, caption IDs, and concepts, then calculates degree distributions and coverage.


In [ ]:
video_to_ccid = {}
ccid_to_video_count = Counter()
bad_mapping_lines = 0
with source_path('relations/video_id-ccid.txt').open('r', encoding='utf-8', errors='replace') as f:
    for line in tqdm(f, desc='video_id → ccid'):
        a,b = parse_pair_line(line)
        if a and b:
            video_to_ccid[a] = b; ccid_to_video_count[b] += 1
        else: bad_mapping_lines += 1

concept_degree, concept_video_degree = Counter(), Counter()
concept_video_edges = 0
bad_concept_video_lines = 0
concept_video_sample = []
with source_path('relations/concept-video.txt').open('r', encoding='utf-8', errors='replace') as f:
    for line in tqdm(f, desc='concept ↔ video'):
        a,b = parse_pair_line(line)
        if not a or not b:
            bad_concept_video_lines += 1; continue
        # Detect direction by ID prefix.
        if a.startswith('K_'):
            concept_id, video_id = a, b
        elif b.startswith('K_'):
            concept_id, video_id = b, a
        else:
            concept_id, video_id = a, b
        concept_degree[concept_id] += 1
        concept_video_degree[video_id] += 1
        concept_video_edges += 1
        if len(concept_video_sample) < 5: concept_video_sample.append((concept_id,video_id))

relation_summary = pd.DataFrame([
    {'relation':'video_id-ccid','edges':len(video_to_ccid),'left_nodes':len(video_to_ccid),'right_nodes':len(ccid_to_video_count),'bad_lines':bad_mapping_lines},
    {'relation':'concept-video','edges':concept_video_edges,'left_nodes':len(concept_degree),'right_nodes':len(concept_video_degree),'bad_lines':bad_concept_video_lines},
])
relation_summary.to_csv(TABLE_ROOT/'relation_summary.csv', index=False)
pd.DataFrame(concept_degree.most_common(), columns=['concept_id','video_count']).to_parquet(TABLE_ROOT/'concept_degrees.parquet', index=False)
pd.DataFrame(concept_video_degree.most_common(), columns=['video_or_ccid','concept_count']).to_parquet(TABLE_ROOT/'video_concept_degrees.parquet', index=False)
display(relation_summary)
print('Concept-video sample:', concept_video_sample)


## 9. Complete user-video behavioural EDA

This is the longest analytical cell. It scans the complete 3 GB interaction file once and writes compact Parquet tables. It computes content watched, estimated real playback time, segment counts, playback-speed usage, activity dates, repeated videos, and video popularity.


In [ ]:
USER_PARQUET = TABLE_ROOT/'user_video_user_metrics.parquet'
VIDEO_PARQUET = TABLE_ROOT/'user_video_video_metrics.parquet'

if FORCE_REBUILD or not (USER_PARQUET.exists() and VIDEO_PARQUET.exists()):
    schema = pa.schema([
        ('user_id', pa.string()), ('video_events', pa.int32()), ('unique_videos', pa.int32()),
        ('segments', pa.int32()), ('content_seconds', pa.float64()), ('playback_seconds', pa.float64()),
        ('first_timestamp', pa.int64()), ('last_timestamp', pa.int64()),
    ])
    writer = pq.ParquetWriter(USER_PARQUET, schema, compression='snappy')
    batch, batch_size = [], 50000
    video_events = Counter(); video_users = Counter(); video_content_seconds = Counter()
    speed_counts = Counter(); invalid_segments = 0; missing_video_ids = 0
    total_users = total_video_events = total_segments = 0

    try:
        for obj in tqdm(stream_json(source_path('relations/user-video.json')), desc='Users'):
            uid = obj.get('user_id'); seq = obj.get('seq') or []
            total_users += 1; seen = set(); content_s = playback_s = 0.0; seg_n = 0; timestamps = []
            for event in seq:
                vid = event.get('video_id')
                if not vid: missing_video_ids += 1; continue
                video_events[vid] += 1; seen.add(vid); total_video_events += 1
                event_content = 0.0
                for seg in event.get('segment') or []:
                    try:
                        start, end = float(seg.get('start_point')), float(seg.get('end_point'))
                        speed = float(seg.get('speed') or 1.0)
                        ts = int(seg.get('local_start_time')) if seg.get('local_start_time') is not None else None
                        duration = end-start
                        if duration < 0 or speed <= 0 or not math.isfinite(duration):
                            invalid_segments += 1; continue
                        content_s += duration; event_content += duration; playback_s += duration/speed; seg_n += 1
                        speed_counts[round(speed,2)] += 1
                        if ts: timestamps.append(ts)
                    except (TypeError, ValueError, OverflowError): invalid_segments += 1
                video_content_seconds[vid] += event_content
            for vid in seen: video_users[vid] += 1
            total_segments += seg_n
            batch.append({
                'user_id':str(uid) if uid is not None else None,
                'video_events':len(seq), 'unique_videos':len(seen), 'segments':seg_n,
                'content_seconds':content_s, 'playback_seconds':playback_s,
                'first_timestamp':min(timestamps) if timestamps else None,
                'last_timestamp':max(timestamps) if timestamps else None,
            })
            if len(batch) >= batch_size:
                writer.write_table(pa.Table.from_pylist(batch, schema=schema)); batch.clear()
        if batch: writer.write_table(pa.Table.from_pylist(batch, schema=schema))
    finally:
        writer.close()

    video_metrics = pd.DataFrame({
        'video_id':list(video_events.keys()),
        'events':[video_events[x] for x in video_events],
        'users':[video_users[x] for x in video_events],
        'content_seconds':[video_content_seconds[x] for x in video_events],
    })
    video_metrics['ccid'] = video_metrics.video_id.map(video_to_ccid)
    video_metrics.to_parquet(VIDEO_PARQUET, index=False)
    pd.DataFrame(speed_counts.items(), columns=['speed','segments']).sort_values('speed').to_csv(TABLE_ROOT/'playback_speed_counts.csv', index=False)
    json.dump({'users':total_users,'video_events':total_video_events,'segments':total_segments,'invalid_segments':invalid_segments,'missing_video_ids':missing_video_ids}, open(REPORT_ROOT/'user_video_scan_summary.json','w'), indent=2)
else:
    print('Using existing user-video EDA checkpoints. Set FORCE_REBUILD=True to rescan.')

user_metrics = pd.read_parquet(USER_PARQUET)
video_metrics = pd.read_parquet(VIDEO_PARQUET)
speed_df = pd.read_csv(TABLE_ROOT/'playback_speed_counts.csv')
scan_summary = json.load(open(REPORT_ROOT/'user_video_scan_summary.json'))
display(pd.DataFrame([scan_summary]))
display(user_metrics.describe(percentiles=[.25,.5,.75,.9,.95,.99]))
display(video_metrics.describe(percentiles=[.25,.5,.75,.9,.95,.99]))


## 10. Behavioural charts and long-tail analysis

In [ ]:
plot_users = user_metrics.sample(min(250000, len(user_metrics)), random_state=RANDOM_SEED)
plt.figure(figsize=(9,5)); sns.histplot(np.log1p(plot_users.video_events), bins=60)
plt.xlabel('log(1 + video events per user)'); plt.title('User activity distribution'); savefig('user_activity_log_distribution.png')

plt.figure(figsize=(9,5)); sns.histplot(np.log1p(plot_users.content_seconds/60), bins=60)
plt.xlabel('log(1 + watched content minutes per user)'); plt.title('User watch-time distribution'); savefig('user_watch_time_log_distribution.png')

ranked = video_metrics.sort_values('events', ascending=False).reset_index(drop=True)
ranked['rank'] = np.arange(1,len(ranked)+1)
plt.figure(figsize=(9,5)); plt.loglog(ranked['rank'], ranked['events'].clip(lower=1))
plt.xlabel('Video popularity rank'); plt.ylabel('Viewing events'); plt.title('Long-tail video popularity'); savefig('video_popularity_long_tail.png')

plt.figure(figsize=(9,5)); sns.barplot(data=speed_df.sort_values('segments',ascending=False).head(15), x='speed', y='segments', color='#4C72B0')
plt.xticks(rotation=45); plt.title('Most common playback speeds'); savefig('playback_speed_distribution.png')

top_videos = ranked.head(1000).copy()
top_videos.to_csv(TABLE_ROOT/'top_1000_videos_by_events.csv', index=False)
display(top_videos.head(20))


## 11. Join coverage and short-video suitability

This checks whether behavioural video IDs can be converted to caption IDs and linked to concepts. It then creates a candidate table for videos of 10 minutes or less.


In [ ]:
caption_ids = set(video_df.ccid.dropna().astype(str))
behavior_video_ids = set(video_metrics.video_id.dropna().astype(str))
mapped_behavior = video_metrics.ccid.notna()
mapped_ccids = set(video_metrics.loc[mapped_behavior,'ccid'].dropna().astype(str))
concept_keys = set(concept_video_degree.keys())

# concept-video may use video_id or ccid; choose whichever gives stronger coverage.
concept_matches_video_id = len(concept_keys & behavior_video_ids)
concept_matches_ccid = len(concept_keys & caption_ids)
concept_key_type = 'video_id' if concept_matches_video_id >= concept_matches_ccid else 'ccid'

coverage = pd.DataFrame([
    {'check':'behavior video IDs mapped to ccid','matched':int(mapped_behavior.sum()),'total':len(video_metrics)},
    {'check':'mapped ccids found in video entities','matched':len(mapped_ccids & caption_ids),'total':len(mapped_ccids)},
    {'check':'concept relation keys matching video_id','matched':concept_matches_video_id,'total':len(concept_keys)},
    {'check':'concept relation keys matching ccid','matched':concept_matches_ccid,'total':len(concept_keys)},
])
coverage['coverage_pct'] = 100*coverage.matched/coverage.total.replace(0,np.nan)
coverage.to_csv(TABLE_ROOT/'id_join_coverage.csv', index=False)
display(coverage)
print('Detected concept-video key type:', concept_key_type)

candidate = video_metrics.merge(video_df, on='ccid', how='left')
if concept_key_type == 'video_id':
    candidate['concept_count'] = candidate.video_id.map(concept_video_degree).fillna(0).astype(int)
else:
    candidate['concept_count'] = candidate.ccid.map(concept_video_degree).fillna(0).astype(int)
candidate['watch_coverage_proxy'] = candidate.content_seconds / candidate.duration_seconds.replace(0,np.nan)
candidate['is_short_10min'] = candidate.duration_seconds.between(1,600)
candidate['eligible_initial'] = candidate.is_short_10min & candidate.name.notna() & (candidate.concept_count>0) & (candidate.users>=2)
candidate.to_parquet(TABLE_ROOT/'video_candidate_eda.parquet', index=False)
candidate[candidate.eligible_initial].to_parquet(TABLE_ROOT/'initial_short_video_candidates.parquet', index=False)

display(candidate[['events','users','content_seconds','duration_seconds','subtitle_sentences','concept_count','watch_coverage_proxy','is_short_10min','eligible_initial']].describe())
print('Initial eligible short videos:', int(candidate.eligible_initial.sum()))


## 12. Concept graph charts

In [ ]:
concept_deg_df = pd.read_parquet(TABLE_ROOT/'concept_degrees.parquet')
video_concept_deg_df = pd.read_parquet(TABLE_ROOT/'video_concept_degrees.parquet')

plt.figure(figsize=(9,5)); sns.histplot(np.log1p(concept_deg_df.video_count), bins=60)
plt.xlabel('log(1 + linked videos per concept)'); plt.title('Concept degree distribution'); savefig('concept_degree_distribution.png')

plt.figure(figsize=(9,5)); sns.histplot(video_concept_deg_df.concept_count.clip(upper=video_concept_deg_df.concept_count.quantile(.99)), bins=50)
plt.xlabel('Concepts linked per video'); plt.title('Concepts per video'); savefig('concepts_per_video.png')

concept_deg_df.head(1000).to_csv(TABLE_ROOT/'top_1000_concepts_by_video_count.csv', index=False)
display(concept_deg_df.head(20))


## 13. Prerequisite graph EDA

The three domain files are streamed and inspected without assuming a single schema. The output records field availability and extracts plausible source–target pairs when present.


In [ ]:
def find_edge(record):
    if not isinstance(record, dict): return None, None
    pairs = [('prerequisite','concept'),('prerequisite_id','concept_id'),('source','target'),('src','dst'),('head','tail'),('from','to')]
    for a,b in pairs:
        if a in record and b in record: return str(record[a]), str(record[b])
    vals = [v for v in record.values() if isinstance(v,(str,int))]
    return (str(vals[0]),str(vals[1])) if len(vals)>=2 else (None,None)

prereq_rows=[]
for domain in ['psy','cs','math']:
    count=0; bad=0; nodes=set(); fields=Counter(); sample=[]
    for rec in tqdm(stream_json(source_path(f'prerequisites/{domain}.json')), desc=f'Prerequisites {domain}'):
        count += 1
        if isinstance(rec,dict): fields.update(rec.keys())
        a,b=find_edge(rec)
        if a and b: nodes.update([a,b])
        else: bad += 1
        if len(sample)<3: sample.append(rec)
    prereq_rows.append({'domain':domain,'records':count,'extracted_nodes':len(nodes),'unparsed_records':bad,'fields':','.join(fields.keys())})
    with (REPORT_ROOT/f'prerequisite_{domain}_sample.json').open('w',encoding='utf-8') as f: json.dump(sample,f,ensure_ascii=False,indent=2)

prereq_summary=pd.DataFrame(prereq_rows)
prereq_summary.to_csv(TABLE_ROOT/'prerequisite_summary.csv',index=False)
display(prereq_summary)
plt.figure(figsize=(8,5)); sns.barplot(data=prereq_summary,x='domain',y='records',color='#55A868'); plt.title('Prerequisite records by domain'); savefig('prerequisite_records.png')


## 14. Missingness and data-quality report

In [ ]:
quality_checks = [
    ('Required raw files present', not missing),
    ('No partial download markers', not partial),
    ('Video entity IDs available', video_df.ccid.notna().all()),
    ('Positive video durations available', int(video_df.duration_seconds.gt(0).sum()) > 0),
    ('User-video records parsed', scan_summary.get('users',0) > 0),
    ('Video ID mapping available', len(video_to_ccid) > 0),
    ('Concept-video edges available', concept_video_edges > 0),
    ('Behavior IDs map to captions', mapped_behavior.mean() > .5),
    ('Initial short-video candidates available', candidate.eligible_initial.sum() > 0),
]
quality_df = pd.DataFrame(quality_checks, columns=['check','passed'])
quality_df.to_csv(TABLE_ROOT/'data_quality_checks.csv',index=False)
display(quality_df.style.applymap(lambda x: 'background-color:#c6efce' if x is True else ('background-color:#ffc7ce' if x is False else '')))

plt.figure(figsize=(10,5)); missplot=entity_missing.copy(); missplot['label']=missplot.entity+': '+missplot.field
sns.barplot(data=missplot.sort_values('missing_pct',ascending=False),y='label',x='missing_pct',color='#C44E52')
plt.xlabel('Missing (%)'); plt.ylabel(''); plt.title('Entity field missingness'); savefig('entity_missingness.png')


## 15. Save final numerical summaries

In [ ]:
user_quantiles = user_metrics[['video_events','unique_videos','segments','content_seconds','playback_seconds']].quantile([0,.25,.5,.75,.9,.95,.99,1])
video_quantiles = video_metrics[['events','users','content_seconds']].quantile([0,.25,.5,.75,.9,.95,.99,1])
duration_quantiles = video_df[['duration_seconds','subtitle_sentences','subtitle_characters']].quantile([0,.25,.5,.75,.9,.95,.99,1])
user_quantiles.to_csv(TABLE_ROOT/'user_metric_quantiles.csv')
video_quantiles.to_csv(TABLE_ROOT/'behavior_video_metric_quantiles.csv')
duration_quantiles.to_csv(TABLE_ROOT/'video_entity_quantiles.csv')

summary = {
    'created_at': datetime.now(timezone.utc).isoformat(),
    'raw_root': str(RAW_ROOT), 'output_root': str(OUT_ROOT),
    'entity_counts': entity_summary.set_index('entity').records.to_dict(),
    'user_video': scan_summary,
    'video_id_ccid_edges': len(video_to_ccid),
    'concept_video_edges': concept_video_edges,
    'concepts_linked': len(concept_degree),
    'videos_or_ccids_linked_to_concepts': len(concept_video_degree),
    'detected_concept_video_key_type': concept_key_type,
    'short_videos_10min': int(video_df.duration_seconds.between(1,600).sum()),
    'initial_eligible_short_videos': int(candidate.eligible_initial.sum()),
    'all_quality_checks_passed': bool(quality_df.passed.all()),
}
with (REPORT_ROOT/'eda_summary.json').open('w',encoding='utf-8') as f: json.dump(summary,f,ensure_ascii=False,indent=2)
display(pd.json_normalize(summary))


## 16. Generate a readable EDA report

In [ ]:
report = f'''# MOOCCubeX EDA Report

Generated: {summary['created_at']}

## Dataset scale
- Users in profile data: {summary['entity_counts'].get('user',0):,}
- Videos/caption records: {summary['entity_counts'].get('video',0):,}
- Courses: {summary['entity_counts'].get('course',0):,}
- Concepts: {summary['entity_counts'].get('concept',0):,}
- Users with video behaviour: {summary['user_video'].get('users',0):,}
- Video viewing events: {summary['user_video'].get('video_events',0):,}
- Watch segments: {summary['user_video'].get('segments',0):,}

## Knowledge graph
- Video-ID-to-caption-ID links: {summary['video_id_ccid_edges']:,}
- Concept-video edges: {summary['concept_video_edges']:,}
- Concepts connected to videos: {summary['concepts_linked']:,}
- Detected video key in concept relations: {summary['detected_concept_video_key_type']}

## Short-video recommendation readiness
- Video entities no longer than 10 minutes: {summary['short_videos_10min']:,}
- Initial candidates with metadata, concepts and at least two users: {summary['initial_eligible_short_videos']:,}
- All automated quality checks passed: {summary['all_quality_checks_passed']}

## Recommended preprocessing stage
1. Filter valid users and videos using the saved candidate table.
2. Convert watch segments into implicit engagement labels.
3. Join video IDs to caption IDs and concept IDs.
4. Build chronological train, validation and test splits.
5. Construct the concept graph from concept-video and prerequisite relations.
6. Save model-ready Parquet tables under `/content/drive/MyDrive/DataCon/processed`.
'''

report_path = REPORT_ROOT/'MOOCCubeX_EDA_Report.md'
report_path.write_text(report,encoding='utf-8')
print(report)
print('Report saved:',report_path)
print('Tables:',TABLE_ROOT)
print('Figures:',FIG_ROOT)


## EDA complete

The next notebook should perform preprocessing and create chronological model splits. Do not delete `EDA_outputs`; its Parquet checkpoints prevent another full scan of the 3 GB behaviour file.
